# EXP-015 원인 분석: 점수 기반 결합이 무엇을 올리고 내렸나

기준(EXP-014, RRF k 20) 대비 정답 상위 순위에 들어온·빠진 질의를 결합 전 목록별 순위와 정규화 점수로 나눠 본다. 가설은 EXP-014에서 남은 결합 손실 53개가 살아난다는 것이고, 위험은 한 목록이 결합을 지배하는 것이다. 판정과 관계없이 쓴다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.index import load_cfg
from rag.retrieve import Retriever, minmax
from rag.search import query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
PREFIX = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256'
TOP = 5

## 질의별 이동

In [3]:
cfg = load_cfg(['+exp=exp015', 'index.scope=full'])
gold = load_gold(cfg, 'dev-full')
base_run = read_run(RUNS / f'{PREFIX}-k20-dev-full.csv')
cand_run = read_run(RUNS / f'{PREFIX}-cc-dev-full.csv')
rb, rc = gold_ranks(base_run, gold, 10), gold_ranks(cand_run, gold, 10)
res = {}
for k in (1, 5):
    hb, hc = rb <= k, rc <= k
    res[k] = {'gain': ~hb & hc, 'loss': hb & ~hc}
    print(f'top{k}: 진입 {int(res[k]["gain"].sum())}, 이탈 {int(res[k]["loss"].sum())}')
moved = rb.fillna(99) != rc.fillna(99)
print('정답 순위가 바뀐 질의', int(moved.sum()))

top1: 진입 91, 이탈 27
top5: 진입 23, 이탈 10
정답 순위가 바뀐 질의 250


## 결합 전 목록별 정답 순위와 정규화 점수

정답 순위가 바뀐 질의와 기준 실패 질의만 본다. 목록(dense, BM25)과 점수는 k·결합 방식과 무관하다.

In [4]:
ret = Retriever(cfg)
qids, text_path = query_list(cfg)
texts = dict(zip(qids, read_texts(text_path, qids), strict=True))
q_all = query_embeddings(cfg, qids, text_path, query_files(cfg)['emb'])
q_emb = dict(zip(qids, q_all, strict=True))
miss_b = ~(rb <= TOP)
sub = sorted(set(moved[moved].index) | set(miss_b[miss_b].index))
tok = dict(zip(sub, ret.query_tokens([texts[q] for q in sub]), strict=True))
n = len(ret.doc_names)
rows, tops = [], {}
for q in sub:
    row = ret.chunk_emb @ q_emb[q]
    d = np.full(n, -np.inf, dtype=row.dtype)
    np.maximum.at(d, ret.chunk_doc, row)
    bm = ret.bm25.scores(tok[q])
    b = np.full(n, -np.inf, dtype=bm.dtype)
    np.maximum.at(b, ret.bm25_doc, bm)
    g = gold.loc[q, 'doc_id']
    rec = {'qid': q}
    for name, s in [('dense', d), ('bm25', b)]:
        order = np.argsort(-s, kind='stable')[: cfg.retriever.fusion.depth]
        docs = ret.doc_names[order].tolist()
        norm = minmax(docs, s[order].tolist())
        rec[f'{name}_rank'] = docs.index(g) + 1 if g in docs else np.nan
        rec[f'{name}_norm'] = norm.get(g, 0.0)
        tops[(q, name)] = docs[0]
    rows.append(rec)
lr = pd.DataFrame(rows).set_index('qid')
lr['base'], lr['cc'] = rb[sub], rc[sub]
print(len(lr), 'queries')

371 queries


In [5]:
for name in ['gain', 'loss']:
    s = lr.loc[[q for q in sub if res[TOP][name][q]]]
    print(f'top{TOP} {name} {len(s)}')
    cols = ['dense_rank', 'bm25_rank', 'dense_norm', 'bm25_norm']
    print(s[cols].median().round(3).to_dict(), '\n')

top5 gain 23
{'dense_rank': 19.0, 'bm25_rank': 1.0, 'dense_norm': 0.239, 'bm25_norm': 1.0} 

top5 loss 10
{'dense_rank': 10.5, 'bm25_rank': 10.5, 'dense_norm': 0.469, 'bm25_norm': 0.503} 



### 남은 결합 손실 53개

기준(EXP-014) 실패 중 dense·BM25 한 목록은 정답을 상위 5위에 둔 질의가 몇 개 살아났는지.

In [6]:
one_list = lr.index[lr[['dense_rank', 'bm25_rank']].min(axis=1) <= TOP]
fusion_loss = [q for q in one_list if miss_b[q]]
print('기준 실패 중 한 목록은 top5', len(fusion_loss))
print('그중 점수 결합에서 top5', int((rc[fusion_loss] <= TOP).sum()))

기준 실패 중 한 목록은 top5 53
그중 점수 결합에서 top5 22


### 한 목록 지배 여부

결합 1위가 dense 1위·BM25 1위와 같은 비율. 기준(RRF k 20)과 비교한다. 한쪽 비율만 크게 늘면 그 목록이 결합을 지배한다.

In [7]:
first = {n_: r[r['rank'] == 1].set_index('qid')['doc_id']
         for n_, r in [('rrf', base_run), ('cc', cand_run)]}
dom = {}
for n_, f in first.items():
    dom[n_] = {lst: np.mean([f[q] == tops[(q, lst)] for q in sub]) for lst in ['dense', 'bm25']}
print(pd.DataFrame(dom).round(3))

         rrf     cc
dense  0.429  0.367
bm25   0.256  0.485


### 잃은 질의

잃은 질의의 목록별 순위와 정규화 점수.

In [8]:
lost = lr.loc[[q for q in sub if res[TOP]['loss'][q]]]
cols = ['dense_rank', 'bm25_rank', 'dense_norm', 'bm25_norm', 'base', 'cc']
print(lost[cols].round(3).to_string())

             dense_rank  bm25_rank  dense_norm  bm25_norm  base    cc
qid                                                                  
046492-0001        91.0        2.0       0.022      0.577   5.0   NaN
046657-0003         9.0       11.0       0.380      0.499   5.0   6.0
046828-0008         3.0       61.0       0.334      0.046   3.0  10.0
047816-0015         6.0       41.0       0.595      0.187   4.0   6.0
048896-0014        12.0        6.0       0.367      0.817   5.0   6.0
049529-0002         7.0       10.0       0.781      0.593   3.0   6.0
050334-0010        14.0       10.0       0.456      0.507   5.0   8.0
050951-0005         3.0        6.0       0.878      0.676   4.0   6.0
051401-0013        14.0       16.0       0.483      0.457   5.0   6.0
051405-0026        12.0       24.0       0.537      0.307   5.0   7.0


## 정리

- 상위 5위 진입 23, 이탈 10(순증 13). 1위는 진입 91, 이탈 27(순증 64)로 1위가 크게 움직였다
- 남은 결합 손실 53개(기준 EXP-014 실패 중 한 목록은 top5) 중 22개가 살아났다. 살아난 질의는 BM25가 정답을 1위로 확신한 경우가 중심이다(중앙 BM25 1위·정규화 1.0, dense 19위·0.24)
- 잃은 10개는 두 목록 모두 중간 순위였다(중앙 dense·BM25 모두 10.5위, 정규화 0.47·0.50). RRF에서 두 목록의 합의로 상위에 있던 정답이 점수 결합에서 밀렸다
- 결합 1위가 BM25 1위와 같은 비율이 25.6% → 48.5%로 늘고 dense 1위와 같은 비율은 42.9% → 36.7%로 줄었다(정답 순위가 바뀐 질의·기준 실패 질의 371개). α 0.5에서도 BM25 쪽이 1위를 더 자주 정한다. BM25 정규화 점수가 상위에서 더 가파르기 때문으로 보인다(26_01: 2위 중앙 BM25 0.47, dense 0.57)